# CD1 — Class 06 · Lab: Cross-validation

In this class, you will practice the **mechanics** of cross-validation: k-fold, `cross_val_score` (which splits train/test), mean ± deviation, stratification and the rare class limit, leave-one-out, repeated k-fold, the group problem (GroupKFold), time (TimeSeriesSplit), and CV with Pipeline (without data leakage).

> The winner's curse and Nested CV are for **Class 07**.

Complete the cells marked with `# TODO`. Run everything from beginning to end.

In [1]:
import numpy as np, pandas as pd
from sklearn.model_selection import (
    train_test_split, cross_val_score, KFold,
    StratifiedKFold, LeaveOneOut, RepeatedStratifiedKFold, GroupKFold, TimeSeriesSplit
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
np.random.seed(0)

## Data (provided)

We read the **`dados_aula06.csv`** dataset (unbalanced, ~10% positive) and **separate a final test set** that DOES NOT enter into cross-validation.

In [2]:
df = pd.read_csv('dados_aula06.csv')
X = df.drop(columns='target').values
y = df['target'].values
X_dev, X_test, y_dev, y_test = train_test_split(X, y, test_size=0.2,
                                                stratify=y, random_state=0)
print('dev:', X_dev.shape, '| test:', X_test.shape, '| positives in dev:', int(y_dev.sum()))

dev: (320, 8) | test: (80, 8) | positives in dev: 34


## Ex 1 — A single split is misleading

Train a `LogisticRegression` with **5 different seeds** for `train_test_split` (on `X_dev, y_dev`) and print the accuracy of each. Notice how the number **fluctuates** just because of the random split.

In [3]:
accuracies = []
for i in range(5):
    X_train, X_val, y_train, y_val = train_test_split(X_dev, y_dev, test_size=0.2, stratify=y_dev, random_state=i)
    model = LogisticRegression(random_state=0, solver='liblinear')
    model.fit(X_train, y_train)
    y_pred = model.predict(X_val)
    accuracy = accuracy_score(y_val, y_pred)
    accuracies.append(accuracy)
    print(f'Accuracy with seed {i}: {accuracy:.4f}')

print(f'\nMean accuracy: {np.mean(accuracies):.4f}')
print(f'Standard deviation of accuracies: {np.std(accuracies):.4f}')

Accuracy with seed 0: 0.8906
Accuracy with seed 1: 0.8750
Accuracy with seed 2: 0.8906
Accuracy with seed 3: 0.8750
Accuracy with seed 4: 0.9219

Mean accuracy: 0.8906
Standard deviation of accuracies: 0.0171


## Ex 2 — k-fold with `cross_val_score`

Run a **5-fold** and print the 5 scores, the **mean**, and the **standard deviation**. (Note: you don't manually split train/test — `cv=5` handles the rotation automatically.)

In [4]:
model = LogisticRegression(random_state=0, solver='liblinear')
scores = cross_val_score(model, X_dev, y_dev, cv=5, scoring='accuracy')
print(f'Accuracies of the 5 folds: {scores}')
print(f'Mean: {scores.mean():.4f}')
print(f'Standard deviation: {scores.std():.4f}')

Accuracies of the 5 folds: [0.90625  0.90625  0.890625 0.875    0.890625]
Mean: 0.8938
Standard deviation: 0.0117


## Ex 3 — Choosing k

Compare **k = 5** and **k = 10**: print mean ± deviation and remember that k=10 performs more training runs.

In [5]:
model = LogisticRegression(random_state=0, solver='liblinear')

# K=5
scores_5_fold = cross_val_score(model, X_dev, y_dev, cv=5, scoring='accuracy')
print(f'K=5: Mean = {scores_5_fold.mean():.4f} \u00B1 {scores_5_fold.std():.4f}')

# K=10
scores_10_fold = cross_val_score(model, X_dev, y_dev, cv=10, scoring='accuracy')
print(f'K=10: Mean = {scores_10_fold.mean():.4f} \u00B1 {scores_10_fold.std():.4f}')

K=5: Mean = 0.8938 ± 0.0117
K=10: Mean = 0.8938 ± 0.0286


## Ex 4 — StratifiedKFold preserves classes

Count how many **positives** fall into each test fold using `KFold` (random) and `StratifiedKFold`. See how stratified balances.

In [6]:
print('Positive count per fold (KFold):')
kf = KFold(n_splits=5, shuffle=True, random_state=0)
for fold, (train_index, test_index) in enumerate(kf.split(X_dev, y_dev)):
    y_test_fold = y_dev[test_index]
    print(f'  Fold {fold+1}: {y_test_fold.sum()} positives')

print('\nPositive count per fold (StratifiedKFold):')
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
for fold, (train_index, test_index) in enumerate(skf.split(X_dev, y_dev)):
    y_test_fold = y_dev[test_index]
    print(f'  Fold {fold+1}: {y_test_fold.sum()} positives')

Positive count per fold (KFold):
  Fold 1: 10 positives
  Fold 2: 12 positives
  Fold 3: 4 positives
  Fold 4: 4 positives
  Fold 5: 4 positives

Positive count per fold (StratifiedKFold):
  Fold 1: 6 positives
  Fold 2: 7 positives
  Fold 3: 7 positives
  Fold 4: 7 positives
  Fold 5: 7 positives


## Ex 5 — The rare class limit

Create a dataset with **only 4 positives** and count the positives per fold with `StratifiedKFold` for k = 3, 5, and 10. Confirm the rule **k ≤ number of positives**: from k = 5, folds with **0 positives** appear (sklearn still warns `least populated class...`).

In [7]:
positive_indices = np.where(y_dev == 1)[0]

np.random.seed(0)
negative_indices = np.random.choice(np.where(y_dev == 0)[0], size=len(positive_indices), replace=False)


idx_pos = np.where(y_dev == 1)[0][:4]
idx_neg = np.where(y_dev == 0)[0][:4]

selected_indices = np.concatenate([idx_pos, idx_neg])
np.random.shuffle(selected_indices)

X_rare = X_dev[selected_indices]
y_rare = y_dev[selected_indices]

print(f'Total positives in the rare set: {y_rare.sum()}')

for k in [3, 5, 10]:
    print(f'\nStratifiedKFold with k = {k}:')
    try:
        skf_rare = StratifiedKFold(n_splits=k, shuffle=True, random_state=0)
        for fold, (train_index, test_index) in enumerate(skf_rare.split(X_rare, y_rare)):
            y_test_fold = y_rare[test_index]
            print(f'  Fold {fold+1}: {y_test_fold.sum()} positives')
    except ValueError as e:
        print(f'  Error: {e}')
        print(f'  This demonstrates the rule: n_splits={k} is greater than the number of positives ({y_rare.sum()}). StratifiedKFold cannot guarantee that each test fold has at least one example of the minority class.')

Total positives in the rare set: 4

StratifiedKFold with k = 3:
  Fold 1: 2 positives
  Fold 2: 1 positives
  Fold 3: 1 positives

StratifiedKFold with k = 5:
  Error: n_splits=5 cannot be greater than the number of members in each class.
  This demonstrates the rule: n_splits=5 is greater than the number of positives (4). StratifiedKFold cannot guarantee that each test fold has at least one example of the minority class.

StratifiedKFold with k = 10:
  Error: Cannot have number of splits n_splits=10 greater than the number of samples: n_samples=8.
  This demonstrates the rule: n_splits=10 is greater than the number of positives (4). StratifiedKFold cannot guarantee that each test fold has at least one example of the minority class.


## Ex 6 — Leave-One-Out

Run **LOO** on a subset of 40 examples. Show that the number of rounds is n, that each score is **0 or 1**, and the average accuracy.

In [8]:
n_subset = 40
X_loo = X_dev[:n_subset]
y_loo = y_dev[:n_subset]

loo = LeaveOneOut()
model = LogisticRegression(random_state=0, solver='liblinear')

scores_loo = cross_val_score(model, X_loo, y_loo, cv=loo, scoring='accuracy')

print(f'Number of rounds (samples in X_loo): {len(scores_loo)}')
print(f'First 10 scores (0 or 1): {scores_loo[:10]}')
print(f'Mean accuracy: {scores_loo.mean():.4f}')

Number of rounds (samples in X_loo): 40
First 10 scores (0 or 1): [1. 0. 1. 1. 1. 1. 1. 1. 1. 1.]
Mean accuracy: 0.9000


## Ex 7 — RepeatedKFold

Use `RepeatedStratifiedKFold` (5 folds × 10 repetitions = 50 scores). Print the mean and standard deviation — more stable than a single repetition.

In [9]:
model = LogisticRegression(random_state=0, solver='liblinear')
rskf = RepeatedStratifiedKFold(n_splits=5, n_repeats=10, random_state=0)
scores_rskf = cross_val_score(model, X_dev, y_dev, cv=rskf, scoring='accuracy')

print(f'Mean accuracies (50 rounds): {scores_rskf.mean():.4f}')
print(f'Standard deviation of accuracies (50 rounds): {scores_rskf.std():.4f}')

Mean accuracies (50 rounds): 0.8944
Standard deviation of accuracies (50 rounds): 0.0131


## Ex 8 — The group problem and GroupKFold

Create `groups` (blocks of 5 rows = one 'patient'). Show that random `KFold` puts the **same group** on both sides (leaks) and `GroupKFold` **does not**.

In [10]:
n_samples = len(X_dev)
groups = np.repeat(np.arange(n_samples // 5), 5)[:n_samples]

print('KFold (leakage):')
kf = KFold(n_splits=5, shuffle=True, random_state=0)
for fold, (train_index, test_index) in enumerate(kf.split(X_dev, y_dev, groups)):
    train_groups = set(groups[train_index])
    test_groups = set(groups[test_index])
    if len(train_groups.intersection(test_groups)) > 0:
        print(f'  Fold {fold+1}: Leaking groups: {train_groups.intersection(test_groups)}', f'({len(train_groups.intersection(test_groups))} groups)') # Added count of leaking groups
    else:
        print(f'  Fold {fold+1}: No group leakage (unexpected with KFold)')

print('\nGroupKFold (no leakage):')
gkf = GroupKFold(n_splits=5)
for fold, (train_index, test_index) in enumerate(gkf.split(X_dev, y_dev, groups)):
    train_groups = set(groups[train_index])
    test_groups = set(groups[test_index])
    if len(train_groups.intersection(test_groups)) > 0:
        print(f'  Fold {fold+1}: Leaking groups: {train_groups.intersection(test_groups)} (ERROR - should not leak)')
    else:
        print(f'  Fold {fold+1}: No group leakage.')

KFold (leakage):
  Fold 1: Leaking groups: {np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(16), np.int64(18), np.int64(20), np.int64(21), np.int64(24), np.int64(26), np.int64(27), np.int64(28), np.int64(30), np.int64(31), np.int64(32), np.int64(33), np.int64(34), np.int64(35), np.int64(36), np.int64(40), np.int64(41), np.int64(43), np.int64(45), np.int64(46), np.int64(48), np.int64(49), np.int64(50), np.int64(51), np.int64(52), np.int64(53), np.int64(54), np.int64(55), np.int64(57), np.int64(61), np.int64(62), np.int64(63)} (43 groups)
  Fold 2: Leaking groups: {np.int64(1), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(14), np.int64(15), np.int64(17), np.int64(19), np.int64(20), np.int64(21), np.int64(23), np.int64(24), np.int64(25), np.int64(26), np.int64(27), np.int64(28), np.int64(31), np.int64(32), np.int64(34), np.int

/usr/local/lib/python3.13/dist-packages/sklearn/model_selection/_split.py:86: UserWarning: The groups parameter is ignored by KFold
  warnings.warn(


## Ex 9 — Time: TimeSeriesSplit

Show, fold by fold, that the training set is always the **past** and the test set is the **future** (the highest training index is less than the lowest test index).

In [11]:
from sklearn.model_selection import TimeSeriesSplit

tscv = TimeSeriesSplit(n_splits=5)

print('TimeSeriesSplit:')
for fold, (train_index, test_index) in enumerate(tscv.split(X_dev)):
    print(f'  Fold {fold+1}:')
    print(f'    Max train index: {train_index.max()}')
    print(f'    Min test index: {test_index.min()}')
    assert train_index.max() < test_index.min() # Confirms that train is always before test
    print(f'    Train has {len(train_index)} samples, Test has {len(test_index)} samples')

TimeSeriesSplit:
  Fold 1:
    Max train index: 54
    Min test index: 55
    Train has 55 samples, Test has 53 samples
  Fold 2:
    Max train index: 107
    Min test index: 108
    Train has 108 samples, Test has 53 samples
  Fold 3:
    Max train index: 160
    Min test index: 161
    Train has 161 samples, Test has 53 samples
  Fold 4:
    Max train index: 213
    Min test index: 214
    Train has 214 samples, Test has 53 samples
  Fold 5:
    Max train index: 266
    Min test index: 267
    Train has 267 samples, Test has 53 samples


## Ex 10 — Why pre-processing must stay INSIDE CV

Any step that **learns from the data** (feature selection, standardization, imputation) needs to be redone **in each fold, only with the training data**. To highlight data leakage, we add **200 columns of pure noise** and use feature selection. Compare selecting **before** CV (looking at the entire dataset) with selecting **inside** the `Pipeline`.

In [12]:
from sklearn.feature_selection import SelectKBest, f_classif

# Add 200 columns of noise
np.random.seed(0)
X_dev_noisy = np.hstack([X_dev, np.random.randn(X_dev.shape[0], 200)])

# Model and CV
model = LogisticRegression(random_state=0, solver='liblinear')
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)

print('Scenario 1: Feature selection BEFORE CV (data leakage)')
# Select the 5 best features from the entire X_dev_noisy
selector_pre_cv = SelectKBest(f_classif, k=5)
X_dev_selected_pre_cv = selector_pre_cv.fit_transform(X_dev_noisy, y_dev)

scores_pre_cv = cross_val_score(model, X_dev_selected_pre_cv, y_dev, cv=skf, scoring='accuracy')
print(f'  Mean (pre-CV): {scores_pre_cv.mean():.4f} \u00B1 {scores_pre_cv.std():.4f}')

print('\nScenario 2: Feature selection INSIDE the Pipeline (no leakage)')
# Pipeline that includes feature selection and the model
pipeline_cv = Pipeline([
    ('selector', SelectKBest(f_classif, k=5)),
    ('classifier', LogisticRegression(random_state=0, solver='liblinear'))
])

scores_in_cv = cross_val_score(pipeline_cv, X_dev_noisy, y_dev, cv=skf, scoring='accuracy')
print(f'  Mean (inside Pipeline): {scores_in_cv.mean():.4f} \u00B1 {scores_in_cv.std():.4f}')

print('\nNote that the accuracy in scenario 1 is artificially high due to leakage.')

Scenario 1: Feature selection BEFORE CV (data leakage)
  Mean (pre-CV): 0.8906 ± 0.0140

Scenario 2: Feature selection INSIDE the Pipeline (no leakage)
  Mean (inside Pipeline): 0.8812 ± 0.0125

Note that the accuracy in scenario 1 is artificially high due to leakage.


## Ex 11 — The final score, on the held-out test set

Assemble the `Pipeline` (scaler + model), train on the entire `X_dev`, and measure **only once** on `X_test`. This is the honest number you report.

In [13]:
pipeline_final = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression(random_state=0, solver='liblinear'))
])

# Train the pipeline on the entire development set (X_dev, y_dev)
pipeline_final.fit(X_dev, y_dev)

# Make predictions on the final test set (X_test)
y_pred_test = pipeline_final.predict(X_test)

# Calculate the accuracy on the test set
final_accuracy = accuracy_score(y_test, y_pred_test)

print(f'Final accuracy on the test set: {final_accuracy:.4f}')

Final accuracy on the test set: 0.8875


---
**Conclusion.** Class insight: a single split has variance; CV replaces this with the average of several. Choose the scheme based on the data (stratified in classification, GroupKFold with repetition per entity, TimeSeriesSplit with date) and always run it within a Pipeline.